# smoke regression
regression of gpp on light, diffuse fraction and weather at each site, smoke attribution with a day block bootstrap, and the modelled radiation effect of smoke and sulfate (figures 2 and 3)

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import statsmodels.api as sm

data, out = Path("data"), Path("results")
out.mkdir(exist_ok=True)
drivers = ["PAR", "DF", "COSZ", "VPD", "TA", "SWC"]


def design(v):
    x = pd.DataFrame({k: np.asarray(v[k], float) for k in drivers})
    x["PAR"] = x.PAR / 1000
    x["PAR2"] = x.PAR ** 2
    x["PARxDF"] = x.PAR * x.DF
    return sm.add_constant(x[["PAR", "PAR2", "DF", "PARxDF", "COSZ", "VPD", "TA", "SWC"]], has_constant="add")

sites = {"US-Bi2": "maize", "US-Ne1": "maize", "US-Ne2": "maize", "US-Ne3": "maize",
         "US-xUN": "forest", "US-UMB": "forest", "US-MMS": "forest", "US-Ha1": "forest"}
broadband = ["US-xUN"]  # diffuse fraction from a shortwave sensor
light = pd.read_csv(out / "light_states.csv")  # from 1_radiative_transfer
aods = sorted(light.x.unique())

In [2]:
def match(d):
    # each smoke hour against clean clear sky hours within 15 days and 1 hour of the same time
    ref = d[d.is_bg]
    rows = []
    for r in d[d.is_smoke].itertuples():
        w = ref[((ref.date - r.date).dt.days.abs() <= 15) & ((ref.hour - r.hour).abs() <= 1)]
        if len(w) >= 3:
            rows.append({"date": r.date, "GPP_r": w.GPP.mean(), "AOD_s": r.AOD550, "AOD_r": w.AOD550.mean(),
                         **{v + "_s": getattr(r, v) for v in drivers}, **{v + "_r": w[v].mean() for v in drivers}})
    return pd.DataFrame(rows)


def effects(fit, m):
    f = lambda v: fit.predict(design(v)).values
    s = {k: m[k + "_s"].values for k in drivers}
    r = {k: m[k + "_r"].values for k in drivers}
    fss, frr = f(s), f({**s, "PAR": r["PAR"], "DF": r["DF"]})
    fsr, frs = f({**s, "DF": r["DF"]}), f({**s, "PAR": r["PAR"]})
    e = {"light": 0.5 * ((fsr - frr) + (fss - frs)), "diffuse": 0.5 * ((frs - frr) + (fss - fsr))}
    for k, v in (("vpd", "VPD"), ("ta", "TA"), ("swc", "SWC")):
        e[k] = 0.5 * ((fss - f({**s, v: r[v]})) + (f({**r, v: s[v]}) - f(r)))
    e["radiation"] = e["light"] + e["diffuse"]
    e["total"] = e["radiation"] + e["vpd"] + e["ta"] + e["swc"]
    return {k: 100 * x.mean() / m.GPP_r.mean() for k, x in e.items()}


def state(case, band, x, sza):
    # libradtran light state at added aod x, interpolated to each solar zenith angle
    s = light[(light.case == case) & (light.band == band)]
    res = []
    for col in ("frac", "dDF"):
        p = s.pivot(index="sza", columns="x", values=col)
        at_x = np.array([np.interp(x, p.columns, p.loc[z]) for z in p.index])
        res.append(np.interp(np.clip(sza, p.index.min(), p.index.max()), p.index, at_x))
    return res


def curve(fit, m, case, band):
    # radiation effect of added aerosol on the matched clean reference hours
    sza = np.degrees(np.arccos(m.COSZ_r.clip(-1, 1)))
    ref = {k: m[k + "_r"].values for k in drivers}
    g0 = fit.predict(design(ref)).values
    res = []
    for x in aods:
        frac, ddf = state(case, band, x, sza)
        v = {**ref, "PAR": ref["PAR"] * frac, "DF": np.clip(ref["DF"] + ddf, 0, 1)}
        res.append(100 * (fit.predict(design(v)).values - g0).mean() / m.GPP_r.mean())
    return res

In [3]:
bars, curves, observed, points = [], [], [], []
for site, group in sites.items():
    d = pd.read_csv(data / f"site_{site}.csv", parse_dates=["date"])
    m = match(d)
    x, y = design(d), d.GPP.values
    fit = sm.OLS(y, x).fit()
    point = effects(fit, m)
    rng = np.random.default_rng(101)
    fit_days, smoke_days = list(d.groupby("date").indices.values()), list(m.groupby("date").indices.values())
    draws = []
    for _ in range(1000):
        i = np.concatenate([fit_days[k] for k in rng.integers(0, len(fit_days), len(fit_days))])
        j = np.concatenate([smoke_days[k] for k in rng.integers(0, len(smoke_days), len(smoke_days))])
        draws.append(effects(sm.OLS(y[i], x.iloc[i]).fit(), m.iloc[j]))
    draws = pd.DataFrame(draws)
    bars += [{"site": site, "n": len(m), "component": k, "pct": v, "lo": np.percentile(draws[k], 2.5),
              "hi": np.percentile(draws[k], 97.5)} for k, v in point.items()]
    band = "BB" if site in broadband else "PAR"
    for case in ("smoke", "sulfate"):
        curves.append(pd.DataFrame({"site": site, "group": group, "case": case, "x": aods, "gpp": curve(fit, m, case, band)}))
    observed.append({"site": site, "group": group, "daod": (m.AOD_s - m.AOD_r).mean(), "pct": point["radiation"],
                     "lo": np.percentile(draws.radiation, 2.5), "hi": np.percentile(draws.radiation, 97.5)})
    if band == "PAR":
        points.append(pd.DataFrame({"par_loss": 100 * (1 - m.PAR_s / m.PAR_r), "dDF": m.DF_s - m.DF_r}))

bars = pd.DataFrame(bars)
bars.to_csv(out / "bar_effects.csv", index=False)
pd.concat(curves).to_csv(out / "radiation_curves.csv", index=False)
pd.DataFrame(observed).to_csv(out / "observed_radiation.csv", index=False)
pd.concat(points).to_csv(out / "smoke_light_points.csv", index=False)
bars.pivot(index="site", columns="component", values="pct").round(1)

Out[3]: 
component  diffuse  light  radiation  swc    ta  total   vpd
site                                                        
US-Bi2         4.6   -3.0        1.6  0.9  -0.6    4.6   2.6
US-Ha1         8.9   -2.5        6.4  0.1   0.7    7.4   0.3
US-MMS         7.7   -3.2        4.5 -0.5  10.1   17.6   3.5
US-Ne1        15.6   -6.8        8.8 -0.4   4.9   18.1   4.8
US-Ne2        17.9   -9.7        8.2 -0.5   5.0   18.9   6.2
US-Ne3        11.7   -6.8        4.9 -1.4   3.1   11.1   4.5
US-UMB         4.9   -1.5        3.4  0.3   0.7    1.5  -2.9
US-xUN        16.7   -7.8        9.0 -0.3  -3.0   18.3  12.6


component,diffuse,light,radiation,swc,ta,total,vpd
site,,,,,,,
US-Bi2,4.6,-3.0,1.6,0.9,-0.6,4.6,2.6
US-Ha1,8.9,-2.5,6.4,0.1,0.7,7.4,0.3
US-MMS,7.7,-3.2,4.5,-0.5,10.1,17.6,3.5
US-Ne1,15.6,-6.8,8.8,-0.4,4.9,18.1,4.8
US-Ne2,17.9,-9.7,8.2,-0.5,5.0,18.9,6.2
US-Ne3,11.7,-6.8,4.9,-1.4,3.1,11.1,4.5
US-UMB,4.9,-1.5,3.4,0.3,0.7,1.5,-2.9
US-xUN,16.7,-7.8,9.0,-0.3,-3.0,18.3,12.6


In [4]:
# light environment curves at 30 degrees zenith and the pinatubo equivalent sulfate aod
# gu et al. 2003 fig 1: direct normal and diffuse shortwave at 70 degrees solar elevation, and noon gpp change
gu_rad = {"normal": (939, 76), "1992": (893, 140), "1993": (915, 95)}
gu_gpp = {"1992": 23, "1993": 8}
el = np.sin(np.radians(70))
gdf = {k: dif / (dni * el + dif) for k, (dni, dif) in gu_rad.items()}
bb = light[(light.case == "sulfate") & (light.band == "BB") & (light.sza == 20)].sort_values("x")
par = light[(light.band == "PAR") & (light.sza == 30)].sort_values("x").assign(par_loss=lambda t: 100 * (1 - t.frac))
par[["case", "x", "par_loss", "dDF"]].to_csv(out / "light_curves.csv", index=False)
sul = par[par.case == "sulfate"]
pin = []
for yr in ("1992", "1993"):
    a = np.interp(gdf[yr] - gdf["normal"], bb.dDF, bb.x)
    pin.append({"year": yr, "aod": a, "par_loss": np.interp(a, sul.x, sul.par_loss), "dDF": np.interp(a, sul.x, sul.dDF),
                "gpp": gu_gpp[yr]})
pd.DataFrame(pin).to_csv(out / "pinatubo.csv", index=False)
pd.DataFrame(pin)

Out[4]: 
   year       aod  par_loss       dDF  gpp
0  1992  0.086893  0.577905  0.076200   23
1  1993  0.027068  0.173625  0.024528    8


,year,aod,par_loss,dDF,gpp
0,1992,0.086893,0.577905,0.076200,23
1,1993,0.027068,0.173625,0.024528,8
